In [1]:
%load_ext autoreload
%autoreload 2

In [10]:
import numpy as np
import pandas as pd
from fuzzywuzzy import process
from constants import DATA_FILENAME, TORVIK_FILE, KENPOM_FILE, ESPN_IDS, ORDER_OF_REGIONS
from scrape_functions import parse_bracket_data, scrape_espn_data

pd.set_option('expand_frame_repr', False)
pd.set_option('display.max_columns', 0)  # Display any number of columns
pd.set_option('display.max_rows', 68)  # Display any number of rows

In [15]:
YEAR = 2026

In [16]:
# This needs to run after the first four games to get the correct names
GENDERS = ["mens", "womens"]
for gender in GENDERS:
    simple = parse_bracket_data(
        scrape_espn_data(id_=ESPN_IDS[YEAR][gender])
    )[['team_name', 'region_id', 'seed']]
    simple['team_region'] = simple['region_id'].apply(
        lambda x: dict(enumerate(ORDER_OF_REGIONS[gender]))[x - 1]
    )
    simple.to_csv(DATA_FILENAME.format(gender=gender), index=False)

# Add KenPom to Mens

In [69]:
kenpom = pd.read_csv(KENPOM_FILE.format(gender="mens"))
team_info = pd.read_csv(DATA_FILENAME.format(gender="mens"))

In [70]:
# Match Kenpom and ESPN team names
kenpom['team_name'] = np.nan
choices = team_info['team_name'].tolist()  # Get all of the ESPN team names
for index, row in kenpom.iterrows():
    team = row['Team']
    match = process.extractOne(team, choices)  # extract the best match for every kenpom name
    if match[1] == 100:  # it's a perfect match
        espn_name = team_info[team_info['team_name'] == match[0]]['team_name'].values[0]
        kenpom.loc[index, 'team_name'] = espn_name
        choices.remove(match[0])  # remove that team as a possible choice

# Need to do this first, otherwise the >80 match ¯\_(ツ)_/¯
kenpom.loc[kenpom['Team'] == "Ohio St.", 'team_name'] = "Ohio State"
choices.remove("Ohio State")
kenpom.loc[kenpom['Team'] == "N.C. State", 'team_name'] = 'North Carolina State'
# choices.remove("North Carolina State")
kenpom.loc[kenpom['Team'] == 'Iowa St.', 'team_name'] = 'Iowa State'
choices.remove('Iowa State')
kenpom.loc[kenpom['Team'] == 'North Dakota St.', 'team_name'] = 'N Dakota St'
choices.remove('N Dakota St')

scores = {}
for index, row in kenpom[kenpom['team_name'].isna()].iterrows():
    team = row['Team']
    match = process.extractOne(team, choices)
    espn_name = team_info[team_info['team_name'] == match[0]]['team_name'].values[0]
    scores[team] = match, espn_name
    if match[1] > 80:
        kenpom.loc[index, 'team_name'] = espn_name
        choices.remove(match[0])  # remove that team as a possible choice

missing_kenpom = kenpom.loc[kenpom['team_name'].isna(), 'Team'].tolist()
missing_espn = team_info.loc[team_info['team_name'].isin(choices)][['team_name']]

print("Missing Kenpom Teams: {}".format(missing_kenpom))
print("Matching ESPN Teams:\n {}".format(missing_espn))

Missing Kenpom Teams: ['Connecticut', 'Texas', 'Miami OH', 'Howard', 'LIU', 'Lehigh', 'Prairie View A&M']
Matching ESPN Teams:
       team_name
5   Long Island
41       PV/LEH
55     TEX/NCSU
60        UConn


In [71]:
scores

{'Connecticut': (('UConn', 72), 'UConn'),
 "St. John's": (("St John's", 95), "St John's"),
 'Utah St.': (('Utah State', 82), 'Utah State'),
 'Miami FL': (('Miami', 90), 'Miami'),
 'Texas': (('TEX/NCSU', 64), 'TEX/NCSU'),
 'SMU': (('M-OH/SMU', 90), 'M-OH/SMU'),
 'Miami OH': (('Long Island', 40), 'Long Island'),
 'Cal Baptist': (('CA Baptist', 95), 'CA Baptist'),
 'Hawaii': (("Hawai'i", 92), "Hawai'i"),
 'UMBC': (('UMBC/HOW', 90), 'UMBC/HOW'),
 'Howard': (('Long Island', 35), 'Long Island'),
 'LIU': (('Long Island', 30), 'Long Island'),
 'Lehigh': (('PV/LEH', 50), 'PV/LEH'),
 'Prairie View A&M': (('PV/LEH', 43), 'PV/LEH')}

In [72]:
# Need to manually fix the crappy ones
# (kenpom_name, espn_name)
manual_matches = [
    ('Connecticut', 'UConn'),
    ('LIU', 'Long Island'),
    # Temp
    ('Lehigh', 'PV/LEH'),
    ('Texas', 'TEX/NCSU'),
    ('SMU', 'M-OH/SMU'),
    ('UMBC', 'UMBC/HOW'),
]
for team, team_id in manual_matches:
    kenpom.loc[kenpom['Team'] == team, 'team_name'] = team_id

merged = kenpom.merge(team_info, left_on=['team_name', 'Seed'], right_on=['team_name', 'seed'])

simple = merged[['team_region', 'Seed', 'team_name', 'AdjustEM', 'AdjustT']]

In [74]:
simple

,team_region,Seed,team_name,AdjustEM,AdjustT
0,East,1,Duke,38.90,65.3
1,West,1,Arizona,37.66,69.8
2,Midwest,1,Michigan,37.59,70.9
3,South,1,Florida,33.79,70.5
4,South,2,Houston,33.43,63.3
5,Midwest,2,Iowa State,32.42,66.5
6,South,3,Illinois,32.10,65.5
7,West,2,Purdue,31.20,64.4
8,East,3,Michigan St,28.31,66.0
9,West,3,Gonzaga,28.10,68.6


In [75]:
simple.shape

(64, 5)

In [76]:
simple.to_csv(DATA_FILENAME.format(gender="mens"), index=False)

# Add Torvik to Womens

In [77]:
kenpom = pd.read_csv(TORVIK_FILE.format(gender="womens"))
team_info = pd.read_csv(DATA_FILENAME.format(gender="womens"))

In [78]:
kenpom.head()

,Team,Seed,pyth
0,Connecticut,1,0.99961
1,UCLA,1,0.99912
2,Texas,1,0.99883
3,South Carolina,1,0.99864
4,LSU,2,0.99775


In [79]:
team_info.head()

,team_name,region_id,seed,team_region
0,Fairfield,1,11,Regional 1
1,High Point,1,15,Regional 1
2,Vermont,4,14,Regional 3
3,UC San Diego,2,14,Regional 4
4,MOST/SFA,4,16,Regional 3


In [80]:
# Match Kenpom and ESPN team names
kenpom['team_name'] = np.nan
choices = team_info['team_name'].tolist()  # Get all of the ESPN team names
for index, row in kenpom.iterrows():
    team = row['Team']
    match = process.extractOne(team, choices)  # extract the best match for every kenpom name
    if match[1] == 100:  # it's a perfect match
        espn_name = team_info[team_info['team_name'] == match[0]]['team_name'].values[0]
        kenpom.loc[index, 'team_name'] = espn_name
        choices.remove(match[0])  # remove that team as a possible choice

# Need to do this first, otherwise the >80 match ¯\_(ツ)_/¯
kenpom.loc[kenpom['Team'] == "Ohio St.", 'team_name'] = "Ohio State"
choices.remove("Ohio State")
kenpom.loc[kenpom['Team'] == "Mississippi", 'team_name'] = "Ole Miss"
choices.remove("Ole Miss")
# kenpom.loc[kenpom['Team'] == "N.C. State", 'team_name'] = 'North Carolina State'
# # choices.remove("North Carolina State")
kenpom.loc[kenpom['Team'] == 'Iowa St.', 'team_name'] = 'Iowa State'
choices.remove('Iowa State')
# kenpom.loc[kenpom['Team'] == 'North Dakota St.', 'team_name'] = 'N Dakota St'
# choices.remove('N Dakota St')

scores = {}
for index, row in kenpom[kenpom['team_name'].isna()].iterrows():
    team = row['Team']
    match = process.extractOne(team, choices)
    espn_name = team_info[team_info['team_name'] == match[0]]['team_name'].values[0]
    scores[team] = match, espn_name
    if match[1] > 80:
        kenpom.loc[index, 'team_name'] = espn_name
        choices.remove(match[0])  # remove that team as a possible choice

missing_kenpom = kenpom.loc[kenpom['team_name'].isna(), 'Team'].tolist()
missing_espn = team_info.loc[team_info['team_name'].isin(choices)][['team_name']]

print("Missing Kenpom Teams: {}".format(missing_kenpom))
print("Matching ESPN Teams:\n {}".format(missing_espn))

Missing Kenpom Teams: ['Connecticut', 'Nebraska', 'Virginia', 'Richmond', 'Arizona St.', 'Missouri St.', 'Fairleigh Dickinson', 'Stephen F. Austin', 'Southern', 'Samford']
Matching ESPN Teams:
    team_name
4   MOST/SFA
7    SOU/SAM
16     UConn
20       FDU
41  NEB/RICH
59   UVA/ASU


In [81]:
scores

{'Connecticut': (('UConn', 72), 'UConn'),
 'N.C. State': (('NC State', 89), 'NC State'),
 'Nebraska': (('NEB/RICH', 50), 'NEB/RICH'),
 'Virginia': (('W Illinois', 44), 'W Illinois'),
 'Richmond': (('NEB/RICH', 50), 'NEB/RICH'),
 'Arizona St.': (('S Dakota St', 57), 'S Dakota St'),
 'South Dakota St.': (('S Dakota St', 86), 'S Dakota St'),
 'Western Illinois': (('W Illinois', 86), 'W Illinois'),
 'Cal Baptist': (('CA Baptist', 95), 'CA Baptist'),
 'Missouri St.': (('SOU/SAM', 64), 'SOU/SAM'),
 'Fairleigh Dickinson': (('MOST/SFA', 43), 'MOST/SFA'),
 'Stephen F. Austin': (('MOST/SFA', 43), 'MOST/SFA'),
 'Southern': (('SOU/SAM', 40), 'SOU/SAM'),
 'Samford': (('SOU/SAM', 54), 'SOU/SAM')}

In [82]:
# Need to manually fix the crappy ones
# (kenpom_name, espn_name)
manual_matches = [
    ('Connecticut', 'UConn'),
    ('Fairleigh Dickinson', 'FDU'),
    # TEMP
    ('Nebraska', 'NEB/RICH'),
    ('Samford', 'SOU/SAM'),
    ('Missouri St.', 'MOST/SFA'),
    ('Virginia', 'UVA/ASU'),
]
for team, team_id in manual_matches:
    kenpom.loc[kenpom['Team'] == team, 'team_name'] = team_id

merged = kenpom.merge(team_info, left_on=['team_name', 'Seed'], right_on=['team_name', 'seed'])

simple = merged[['team_region', 'Seed', 'team_name', 'pyth']]

In [83]:
simple

,team_region,Seed,team_name,pyth
0,Regional 1,1,UConn,0.999610
1,Regional 2,1,UCLA,0.999120
2,Regional 3,1,Texas,0.998830
3,Regional 4,1,South Carolina,0.998640
4,Regional 2,2,LSU,0.997750
5,Regional 3,2,Michigan,0.995260
6,Regional 2,3,Duke,0.993670
7,Regional 2,4,Minnesota,0.991480
8,Regional 4,2,Iowa,0.990790
9,Regional 1,2,Vanderbilt,0.990610


In [84]:
simple.to_csv(DATA_FILENAME.format(gender="womens"), index=False)

## OLD

In [27]:
# espn = pd.read_csv(ESPN_FILE)
# GENDERS = ["mens", "womens"]
# forecast_dates = espn.groupby(
#     ["gender", "forecast_date", "results_to"]
# ).filter(lambda x: len(x) == 64).groupby("gender")["forecast_date"].unique().to_dict()
# print(forecast_dates)
# results = {}
# for gender in GENDERS:
#     forecast_date = forecast_dates[gender]
#     assert len(forecast_date) == 1
#     espn_slice = (
#         (espn['gender'] == gender) &
#         (espn['forecast_date'] == forecast_date[0]) &
#         (espn["team_alive"] == 1) 
#     )
#     # For backdated runs
#     # espn_slice = (espn['gender'] == 'mens') & (espn['forecast_date'] == "2023-03-15") & (espn["team_alive"] == 1)
#     simple = espn[espn_slice][['team_region', 'team_seed', 'team_name', 'team_id', 'team_rating']].rename(columns={"team_seed": "Seed"})
#     simple["Seed"] = simple["Seed"].str.replace("\D", "", regex=True)
#     results[gender] = simple